# 3. Feature Engineering и модели LightGBM

В предыдущих ноутбуках мы исследовали данные и построили базовые модели. Теперь попробуем улучшить качество предсказаний за счёт новых признаков, описывающих историю взаимодействий пользователей с продуктами.

В рамках ноутбука:

1. Сформируем признаки на основе пользовательских событий
2. Подготовим данные для обучения LightGBM
3. Проверим исходный **LGBMRanker** `interest_v2` для ранжирования продуктов-кандидатов
4. Проверим **LGBMClassifier** с трёхфолдовой кросс-валидацией
5. Сравним результаты и сформируем предсказания для Kaggle

При оценке моделей будем учитывать различия между обучающей и тестовой выборками, из-за которых локальная валидация может не совпадать с результатами на Kaggle.

## 1. Загрузка данных и разделение выборки

Оставим одну из пяти частей для валидации с помощью `StratifiedGroupKFold`. Проверим, что клиенты не пересекаются, и сравним доли продуктов; редкие классы могут разделиться неравномерно.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction import DictVectorizer

PROJECT_DIR = next(path for base in [Path.cwd(), *Path.cwd().parents]
                   for path in [base, base / 'task3']
                   if (path / 'notebooks/03_features_and_lgbm.ipynb').is_file())
DATA_DIR = PROJECT_DIR / 'data'
SUBMITS_DIR = PROJECT_DIR / 'submits'
BATCH_SIZE = 32
RANDOM_STATE = 42

# train и test
train_meta = pq.read_table(DATA_DIR / 'train.parquet',
                          columns=['example_id', 'client_id', 'target_product']).to_pandas()
test_meta = pq.read_table(DATA_DIR / 'test.parquet', columns=['example_id', 'client_id']).to_pandas()

# словарь событий
event_dictionary = pq.read_table(DATA_DIR / 'event_dictionary.parquet').to_pandas()
event_types = event_dictionary.set_index('encoded_event')['event_type'].to_dict()
target_event_codes = set(event_dictionary.loc[event_dictionary['is_target_event'].fillna(False), 'encoded_event'])
action_types = sorted(set(event_types.values()) | {'unknown'})

# разделяем чтобы доли правильных ответов совпадали
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE) 
train_idx, valid_idx = next(splitter.split(
    train_meta, train_meta['target_product'], groups=train_meta['client_id']))
train_idx, valid_idx = np.sort(train_idx), np.sort(valid_idx)

y_train = train_meta.iloc[train_idx]['target_product'].to_numpy()
y_valid = train_meta.iloc[valid_idx]['target_product'].to_numpy()

# проверяем, что клиенты не пересекаются
client_overlap = len(set(train_meta.iloc[train_idx]['client_id']) &
                     set(train_meta.iloc[valid_idx]['client_id']))
assert client_overlap == 0
assert len(train_idx) + len(valid_idx) == len(train_meta)
assert train_meta['example_id'].is_unique and test_meta['example_id'].is_unique
print(f'Обучение: {len(train_idx):,} историй; валидация: {len(valid_idx):,}')
print('Общих клиентов:', client_overlap)

# смотрим на stratify
class_distribution = pd.concat([
    pd.Series(y_train).value_counts().rename('Train, число'),
    pd.Series(y_valid).value_counts().rename('Validation, число'),
], axis=1).fillna(0).astype(int).sort_index()

class_distribution['Train, %'] = class_distribution['Train, число'] / len(y_train) * 100
class_distribution['Validation, %'] = class_distribution['Validation, число'] / len(y_valid) * 100
class_distribution['Разница, п.п.'] = class_distribution['Validation, %'] - class_distribution['Train, %']
top_products = train_meta['target_product'].value_counts().head(10).index
display(class_distribution.loc[top_products].round(2).rename_axis('Продукт'))
print(f'Максимальное отличие долей по всем классам: {class_distribution["Разница, п.п."].abs().max():.3f} п.п.')
print('Классов train / validation:', len(np.unique(y_train)), '/', len(np.unique(y_valid)))
print('Отсутствуют в train / validation:', np.setdiff1d(y_valid, y_train).tolist(), '/',
      np.setdiff1d(y_train, y_valid).tolist())

site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Обучение: 79,999 историй; валидация: 20,001
Общих клиентов: 0


,"Train, число","Validation, число","Train, %","Validation, %","Разница, п.п."
Продукт,,,,,
55,12806,3202,16.01,16.01,0.0
6,11864,2966,14.83,14.83,-0.0
35,11264,2816,14.08,14.08,-0.0
63,8350,2088,10.44,10.44,0.0
58,5799,1450,7.25,7.25,0.0
13,5040,1260,6.30,6.30,-0.0
18,4771,1193,5.96,5.96,0.0
4,3354,839,4.19,4.19,0.0
60,2950,737,3.69,3.68,-0.0


Максимальное отличие долей по всем классам: 0.005 п.п.
Классов train / validation: 32 / 31
Отсутствуют в train / validation: [] / [37]


## 2. Время событий

По описанию данных `timestamp` хранится в миллисекундах. Переводим интервалы в секунды, а длительность истории и давность взаимодействий в часы.

Сам целевой клик и пять событий перед ним скрыты. Давности считаем до конца выданной истории, а не до времени будущего клика.

In [2]:
TIME_SCALE_SECONDS = 0.001

## 3. Создание признаков

Для обучения моделей преобразуем историю взаимодействий каждого пользователя в набор числовых признаков. Попробуем учесть не только частоту действий, но и их последовательность, давность и распределение между продуктами.

Выделим несколько групп признаков:

| Группа | Описание |
|---|---|
| **Общие характеристики** | Длина истории, количество уникальных продуктов, число активных дней и сессий |
| **Типы действий** | Количество и доли кликов, просмотров и целевых действий |
| **Временные характеристики** | Длительность истории, интервалы между событиями, давность последних взаимодействий |
| **Недавняя активность** | Статистика последних 20 и 100 событий, активность за 1, 3 и 7 дней |
| **Интерес к продуктам** | Частота взаимодействий с каждым продуктом, доли кликов и целевых действий, давность интереса |
| **Динамика поведения** | Изменение активности со временем, повторные взаимодействия и переходы между продуктами |

Признаки рассчитываются только по доступной истории пользователя, без использования `target_product`. Для разделения событий на условные сессии используем паузу более 30 минут.

Предполагаем, что сочетание общих характеристик поведения и признаков интереса к конкретным продуктам позволит моделям точнее определять следующий целевой продукт.

In [3]:
def time_features(timestamps):
    known = timestamps[np.isfinite(timestamps)]
    gaps = np.diff(timestamps)
    gaps = gaps[np.isfinite(gaps)] * TIME_SCALE_SECONDS
    return {
        'duration_hours': (known[-1] - known[0]) * TIME_SCALE_SECONDS / 3600 if len(known) > 1 else 0, # общее время истории
        'n_missing_timestamps': len(timestamps) - len(known),                                          # события без времени
        'gap_mean_seconds': gaps.mean() if len(gaps) else 0,                                           # средний промежуток
        'gap_median_seconds': np.median(gaps) if len(gaps) else 0,                                     # мед. пром
        'gap_p90_seconds': np.quantile(gaps, 0.9) if len(gaps) else 0,                                 # 90-ый перц
        'gap_max_seconds': gaps.max() if len(gaps) else 0,                                             # максимальный пром.
    }

In [ ]:

def history_features(events):
    missing = events is None
    events = events or []
    kinds = [event_types.get(e['encoded_event'], 'unknown') if e else 'unknown' for e in events]
    products = [e['encoded_product'] if e else None for e in events]
    target_flags = [e is not None and e['encoded_event'] in target_event_codes for e in events]
    timestamps = np.asarray([e['timestamp'] if e else None for e in events], dtype=np.float64)

    known_times = timestamps[np.isfinite(timestamps)]
    if (np.diff(known_times) < 0).any():
        raise ValueError('Нарушен хронологический порядок событий.')

    end_time = known_times[-1] if len(known_times) else np.nan
    known = [p for p in products if p is not None]
    product_counts = Counter(known)
    counts = Counter(kinds)
    target_products = [p for p, flag in zip(products, target_flags) if flag]
    target_counts = Counter(p for p in target_products if p is not None)
    last_target_indices = {p: i for i, (p, flag) in enumerate(zip(products, target_flags))
                           if flag and p is not None}
    n = len(events)

    # Условные сессии отделяем паузой 30 минут; неизвестное время не создаёт новую сессию.
    session_ids = np.zeros(n, dtype=np.int32)
    previous_time, session = None, 0
    for i, timestamp in enumerate(timestamps):
        if np.isfinite(timestamp):
            if previous_time is not None and (timestamp - previous_time) * TIME_SCALE_SECONDS > 1800:
                session += 1
            previous_time = timestamp
        session_ids[i] = session
    last_session = session_ids[-1] if n else -1
    session_clicks, session_targets = Counter(), Counter()
    click_sessions, target_sessions = {}, {}
    decayed_clicks, decayed_targets = Counter(), Counter()
    for i, (product, kind, flag) in enumerate(zip(products, kinds, target_flags)):
        if product is None:
            continue
        if kind == 'click':
            click_sessions.setdefault(product, set()).add(int(session_ids[i]))
            session_clicks[product] += int(session_ids[i] == last_session)
        if flag:
            target_sessions.setdefault(product, set()).add(int(session_ids[i]))
            session_targets[product] += int(session_ids[i] == last_session)
        if np.isfinite(timestamps[i]) and np.isfinite(end_time):
            # Вес интереса уменьшается вдвое за сутки, считая от конца доступной истории.
            weight = np.exp2(-(end_time - timestamps[i]) * TIME_SCALE_SECONDS / 86400)
            if kind == 'click': decayed_clicks[product] += weight
            if flag: decayed_targets[product] += weight
    recent_interest = {}

    # Общие признаки истории
    result = {
        'history_length': n, 'history_missing': int(missing),                                      # кол-во событий и отсутствие истории
        'n_product_events': len(known), 'known_product_rate': len(known) / max(n, 1),              # кол-во и доля событий с продуктом
        'n_unique_products': len(product_counts),                                                  # кол-во разных продуктов
        'last_product': str(known[-1]) if known else 'missing',                                    # последний известный продукт
        'most_frequent_product': str(min(product_counts, key=lambda p: (-product_counts[p], p)))   # самый частый продукт
                                 if product_counts else 'missing',
    }

    # Сессионная активность дополняет общие счётчики
    result.update({'interest_n_sessions': int(session + 1) if n else 0,                            # кол-во сессий
                   'interest_last_session_length': int((session_ids == last_session).sum()),       # кол-во событий в последней сессии
                   'interest_last_session_n_click': sum(kind == 'click' and sid == last_session    # кол-во кликов в последней сессии
                                                        for kind, sid in zip(kinds, session_ids)),
                   'interest_last_session_n_target': sum(flag and sid == last_session              # кол-во целевых действий в последней сессии
                                                         for flag, sid in zip(target_flags, session_ids))})

    result.update({'n_target_events': len(target_products),                                        # кол-во событий целевого типа
                   'last_target_event_product': str(target_products[-1])
                   if target_products and target_products[-1] is not None else 'missing'})         # продукт последнего такого события

    result.update({f'history_{key}': value for key, value in time_features(timestamps).items()})   # добавляем все временные признаки

    # Количество и доля каждого типа действия
    for kind in action_types:
        result[f'n_{kind}'] = counts[kind]
        result[f'{kind}_rate'] = counts[kind] / max(n, 1)

    # Продукты последних просмотра и клика
    for kind in ['click', 'view']:
        interacted = [p for p, action in zip(products, kinds) if action == kind]
        result[f'last_{kind}_product'] = str(interacted[-1]) if interacted and interacted[-1] is not None else 'missing'

    # Признаки по последним 20 и 100 событиям
    for window in [20, 100]:
        tail_kinds, tail_products = kinds[-window:], products[-window:]
        tail_known = [p for p in tail_products if p is not None]
        tail_counts = Counter(tail_kinds)
        size = len(tail_kinds)
        prefix = f'last{window}_'

        result.update({prefix + 'n_events': size,                                                  # кол-во последних событий
                       prefix + 'n_unique_products': len(set(tail_known)),                         # кол-во разных продуктов
                       prefix + 'known_product_rate': len(tail_known) / max(size, 1)})             # доля событий с продуктом

        result.update({prefix + key: value for key, value in time_features(timestamps[-window:]).items()})  # временные хар-ки

        for kind in action_types:
            result[prefix + f'n_{kind}'] = tail_counts[kind]                                       # кол-во действий данного типа
            result[prefix + f'{kind}_rate'] = tail_counts[kind] / max(size, 1)                     # доля действий данного типа

        window_product_counts = {}

        # Просмотры и клики по каждому продукту в последних событиях
        for kind in ['view', 'click']:
            product_counts_window = Counter(p for p, action in zip(tail_products, tail_kinds)
                                            if p is not None and action == kind)
            window_product_counts[kind] = product_counts_window
            result.update({f'p_{product}_last{window}_{kind}': count
                           for product, count in product_counts_window.items()})

        # События целевого типа по продуктам в последних 20/100 событиях
        target_window_counts = Counter(p for p, flag in zip(tail_products, target_flags[-window:])
                                       if flag and p is not None)
        result.update({f'p_{product}_last{window}_target_event': count
                       for product, count in target_window_counts.items()})

        recent_interest[window] = {**window_product_counts, 'target': target_window_counts, 'size': size}

    clicks = Counter(p for p, kind in zip(products, kinds) if kind == 'click' and p is not None)
    views = Counter(p for p, kind in zip(products, kinds) if kind == 'view' and p is not None)
    recent = Counter(p for p in products[-100:] if p is not None)
    last_indices = {p: i for i, p in enumerate(products) if p is not None}

    # Признаки отдельно для каждого продукта
    for product, count in product_counts.items():
        last_index = last_indices[product]
        time_known = np.isfinite(timestamps[last_index]) and np.isfinite(end_time)
        prefix = f'p_{product}_'

        result.update({
            prefix + 'count': count,                                                               # кол-во взаимодействий с продуктом
            prefix + 'share': count / n,                                                           # доля продукта среди всех событий
            prefix + 'click': clicks[product],                                                     # кол-во кликов
            prefix + 'view': views[product],                                                       # кол-во просмотров
            prefix + 'last100': recent[product],                                                   # упоминания в последних 100 событиях
            prefix + 'last_position': (last_index + 1) / n,                                        # положение последнего взаимодействия
            prefix + 'hours_since_last': (end_time - timestamps[last_index]) * TIME_SCALE_SECONDS / 3600
                                         if time_known else 0,                                     # часов с последнего взаимодействия
            prefix + 'last_time_missing': int(not time_known)                                      # отсутствует время взаимодействия
        })

        last_target_index = last_target_indices.get(product)
        target_time_known = (last_target_index is not None and np.isfinite(timestamps[last_target_index])
                             and np.isfinite(end_time))

        result.update({
            prefix + 'target_event_count': target_counts[product],                                 # кол-во событий целевого типа
            prefix + 'hours_since_last_target_event':
                (end_time - timestamps[last_target_index]) * TIME_SCALE_SECONDS / 3600
                if target_time_known else 0,                                                       # часов с последнего такого события
            prefix + 'last_target_event_time_missing': int(not target_time_known)                  # нет времени такого события
        })

        # Пять добавочных просмотров сглаживают отношения при маленьких счётчиках
        result.update({prefix + 'interest_click_view_ratio': clicks[product] / (views[product] + 5),                 # отношение кликов к просмотрам
                       prefix + 'interest_target_view_ratio': target_counts[product] / (views[product] + 5),         # отношение целевых действий к просмотрам
                       prefix + 'interest_click_share': clicks[product] / max(counts['click'], 1),                   # доля продукта среди всех кликов
                       prefix + 'interest_target_share': target_counts[product] / max(len(target_products), 1)})     # доля продукта среди целевых действий

        for window in [20, 100]:
            window_counts = recent_interest[window]
            result[prefix + f'interest_last{window}_click_view_ratio'] = window_counts['click'][product] / (window_counts['view'][product] + 5)    # отношение кликов к просмотрам в окне
            result[prefix + f'interest_last{window}_target_view_ratio'] = window_counts['target'][product] / (window_counts['view'][product] + 5)  # отношение целевых действий к просмотрам в окне

        # Сравниваем последние 20 событий с предыдущими 80 и недавний интерес со всей историей
        r20, r100 = recent_interest[20], recent_interest[100]
        for action in ['click', 'target']:
            total = clicks[product] if action == 'click' else target_counts[product]
            result[prefix + f'interest_recent_{action}_fraction'] = r100[action][product] / max(total, 1)  # доля действий продукта в последних 100 событиях
            result[prefix + f'interest_{action}_growth'] = (r20[action][product] / max(r20['size'], 1)     # изменение частоты действий в последних 20 событиях
                - (r100[action][product] - r20[action][product]) / max(r100['size'] - r20['size'], 1))

        # Доли с затуханием и повторение интереса в разных сессиях меньше зависят от числа показов
        result.update({prefix + 'interest_decayed_click_share': decayed_clicks[product] / (sum(decayed_clicks.values()) or 1.0),                   # доля кликов с уменьшением веса старых
                       prefix + 'interest_decayed_target_share': decayed_targets[product] / (sum(decayed_targets.values()) or 1.0),                # доля целевых действий с уменьшением веса старых
                       prefix + 'interest_n_click_sessions': len(click_sessions.get(product, ())),                                                 # кол-во сессий с кликами по продукту
                       prefix + 'interest_n_target_sessions': len(target_sessions.get(product, ())),                                               # кол-во сессий с целевыми действиями
                       prefix + 'interest_last_session_click': session_clicks[product],                                                            # кол-во кликов по продукту в последней сессии
                       prefix + 'interest_last_session_target': session_targets[product],                                                          # кол-во целевых действий в последней сессии
                       prefix + 'interest_last_session_click_share': session_clicks[product] / max(sum(session_clicks.values()), 1),               # доля кликов продукта в последней сессии
                       prefix + 'interest_last_session_target_share': session_targets[product] / max(sum(session_targets.values()), 1)})           # доля целевых действий продукта в последней сессии

    # Временные признаки относительно конца доступной истории
    days = np.where(np.isfinite(timestamps) & np.isfinite(end_time),
                    (end_time - timestamps) * TIME_SCALE_SECONDS / 86400, np.nan)
    result.update({
        'v2_end_timestamp_ms': end_time if np.isfinite(end_time) else -1,                                  # время конца истории
        'v2_end_time_missing': int(not n or not np.isfinite(timestamps[-1])),                              # отсутствие времени последнего события
        'v2_n_active_days': len(set((known_times * TIME_SCALE_SECONDS // 86400).astype(np.int64))),        # кол-во активных дней
    })

    event_counts, recent_event_counts = Counter(), Counter()
    positions, timed_actions = {}, {}
    transitions, after_target = Counter(), Counter()
    previous_target = None
    last_target_position = max((i for i, flag in enumerate(target_flags) if flag), default=-1)
    last_target_product = products[last_target_position] if last_target_position >= 0 else None
    timed_counts = {(action, window): Counter() for action in ['click', 'view', 'target'] for window in [1, 3, 7]}
    previous_week = {'click': Counter(), 'target': Counter()}
    global_timed_counts = Counter()
    product_days, product_sessions = {}, {}

    # Один проход: календарные окна, возвраты и переходы между целевыми действиями
    for i, (product, kind, flag) in enumerate(zip(products, kinds, target_flags)):
        if events[i] is not None and kind != 'view':
            event_code = events[i]['encoded_event']
            event_counts[event_code] += 1
            if i >= n - 100:
                recent_event_counts[event_code] += 1
        if np.isfinite(days[i]):
            for window in [1, 3, 7]:
                if days[i] <= window:
                    global_timed_counts['click', window] += int(kind == 'click')
                    global_timed_counts['target', window] += int(flag)
        if product is None:
            if flag:
                previous_target = None
            continue
        product_sessions.setdefault(product, set()).add(int(session_ids[i]))
        if np.isfinite(timestamps[i]):
            product_days.setdefault(product, set()).add(int(timestamps[i] * TIME_SCALE_SECONDS // 86400))
        if i > last_target_position >= 0:
            after_target[product] += 1
        actions = ([kind] if kind in ['click', 'view'] else []) + (['target'] if flag else [])
        for action in actions:
            key = (product, action)
            positions.setdefault(key, []).append(i)
            if np.isfinite(timestamps[i]):
                timed_actions.setdefault(key, []).append(timestamps[i])
            if np.isfinite(days[i]):
                for window in [1, 3, 7]:
                    if days[i] <= window:
                        timed_counts[action, window][product] += 1
                if action != 'view' and 1 < days[i] <= 7:
                    previous_week[action][product] += 1
        if flag:
            if previous_target is not None:
                transitions[previous_target, product] += 1
            previous_target = product

    # Коды действий сохраняют информацию, которая теряется при объединении в view/click/other
    for event_code, count in event_counts.items():
        result[f'v2_event_{event_code}_share'] = count / max(n, 1)                                                      # доля событий данного кода
        result[f'v2_event_{event_code}_last100_share'] = recent_event_counts[event_code] / max(min(n, 100), 1)          # доля кода в последних 100 событиях
    result['v2_events_after_last_target'] = n - last_target_position - 1 if last_target_position >= 0 else n            # кол-во событий после последнего целевого
    result['v2_has_target'] = int(last_target_position >= 0)                                                            # наличие целевых действий в истории
    result['v2_target_product_switches'] = sum(count for (left, right), count in transitions.items() if left != right)  # кол-во смен продукта между целевыми действиями

    for window in [1, 3, 7]:
        result[f'v2_click_{window}d'] = global_timed_counts['click', window]                                            # кол-во кликов за последние дни
        result[f'v2_target_{window}d'] = global_timed_counts['target', window]                                          # кол-во целевых действий за последние дни

    total_windows = {key: sum(value.values()) for key, value in timed_counts.items()}
    rank_values = {(product, action): sum(value > counter[product] for value in counter.values()) / max(len(product_counts) - 1, 1)
                   for action, counter in [('click', clicks), ('target', target_counts)] for product in product_counts}

    # Ранги отражают интерес внутри истории, а не числовые значения кодов продуктов
    for product in product_counts:
        prefix = f'p_{product}_v2_'
        result.update({
            prefix + 'seen': 1,                                                                                                  # наличие продукта в истории
            prefix + 'n_active_days': len(product_days.get(product, ())),                                                        # кол-во дней с продуктом
            prefix + 'session_fraction': len(product_sessions.get(product, ())) / max(session + 1, 1),                           # доля сессий с продуктом
            prefix + 'sessions_since_last': max(last_session - max(product_sessions.get(product, {-1})), 0),                     # кол-во сессий с последнего взаимодействия
            prefix + 'follow_last_target': transitions[last_target_product, product] if last_target_product is not None else 0,  # переходы от последнего целевого продукта
            prefix + 'after_last_target_share': after_target[product] / max(sum(after_target.values()), 1),                      # доля событий продукта после последнего целевого
        })

        for action in ['click', 'view', 'target']:
            indices = positions.get((product, action), [])
            times = timed_actions.get((product, action), [])
            time_ok = bool(indices) and np.isfinite(timestamps[indices[-1]]) and np.isfinite(end_time)
            result[prefix + f'has_{action}'] = int(bool(indices))                                                                          # наличие действия данного типа с продуктом
            result[prefix + f'{action}_time_missing'] = int(not time_ok)                                                                   # отсутствие времени последнего действия
            result[prefix + f'hours_since_{action}'] = (end_time - timestamps[indices[-1]]) * TIME_SCALE_SECONDS / 3600 if time_ok else 0  # часов с последнего действия
            result[prefix + f'steps_since_{action}'] = (n - 1 - indices[-1]) / max(n, 1) if indices else 1                                 # доля событий после последнего действия

            for window in [1, 3, 7]:
                count = timed_counts[action, window][product]
                result[prefix + f'{action}_{window}d'] = count                                                                             # кол-во действий продукта за последние дни
                result[prefix + f'{action}_{window}d_share'] = count / max(total_windows[action, window], 1)                               # доля продукта среди действий за последние дни

            if action in ['click', 'target']:
                gaps = np.diff(times) * TIME_SCALE_SECONDS / 3600
                second_ok = len(indices) > 1 and np.isfinite(timestamps[indices[-2]]) and np.isfinite(end_time)
                result[prefix + f'{action}_rank'] = rank_values[product, action]                                                                        # относительный ранг продукта по числу действий
                result[prefix + f'{action}_acceleration'] = timed_counts[action, 1][product] - previous_week[action][product] / 6                       # активность за сутки минус средняя за 6 дней
                result[prefix + f'{action}_last2_session_share'] = sum(session_ids[j] >= last_session - 1 for j in indices) / max(len(indices), 1)      # доля действий в последних двух сессиях
                result[prefix + f'hours_since_previous_{action}'] = (end_time - timestamps[indices[-2]]) * TIME_SCALE_SECONDS / 3600 if second_ok else 0# часов с предпоследнего действия
                result[prefix + f'previous_{action}_time_missing'] = int(not second_ok)                                                                 # отсутствие времени предпоследнего действия
                result[prefix + f'{action}_gap_mean_hours'] = gaps.mean() if len(gaps) else 0                                                           # средний интервал между действиями в часах
                result[prefix + f'{action}_gap_last_hours'] = gaps[-1] if len(gaps) else 0                                                              # последний интервал между действиями в часах
                result[prefix + f'{action}_gap_count'] = len(gaps)                                                                                      # кол-во интервалов с известным временем

    return result


## 4. Чтение историй по частям

Обрабатываем истории пакетами по 32, чтобы не загружать все данные в память сразу

In [5]:
def feature_rows(path, selected_rows=None):
    selected = set(selected_rows) if selected_rows is not None else None
    offset = 0
    for batch in pq.ParquetFile(path).iter_batches(batch_size=BATCH_SIZE, columns=['events']):
        for i in range(len(batch)):
            if selected is None or offset + i in selected:
                features = history_features(batch.column('events')[i].as_py())
                yield {name: value for name, value in features.items() if isinstance(value, str) or value != 0}
        offset += len(batch)

## 5. Подготовка матриц и кеширование

Преобразуем рассчитанные признаки в разреженные матрицы CSR, которые позволяют эффективнее хранить данные и обучать модели

Чтобы не пересчитывать признаки при каждом запуске, предусмотрим их сохранение и загрузку из кеша. При этом валидационные данные не используются для обучения преобразователя признаков

Код генерации ниже сохранён полностью. Его текстовая сигнатура отличается от исторического кеша. Для воспроизведения опубликованных результатов загружаем кеш только при совпадении сигнатур сохранённых Ranker и KFold, исходных файлов и словаря. `generator_signature` описывает текущий генератор, `signature` — фактически загруженные признаки; они не объявляются одинаковыми. Новый расчёт признаков не подставляется под старые модели автоматически.


In [ ]:
import gc
import re
import inspect
import hashlib
import joblib
import lightgbm as lgb
from time import perf_counter
from scipy.sparse import csr_matrix, vstack
from sklearn.metrics import accuracy_score

RUN_PREPROCESSING = False
RUN_KFOLD = False

USE_CACHE = True
SAVE_CACHE = False
SAVE_SUBMISSIONS = False
SAVE_MODELS = False

# параметры обучения
N_FOLDS = 3
N_THREADS = 2
SAVE_FOLDS = True
RANK_NEGATIVES = 8
RANK_NEGATIVE_STRATEGY = 'mixed'
RANKER_USE_IDENTITY = True

CACHE_DIR = DATA_DIR.parent / 'feature_cache_v4'
CACHE_PATH = CACHE_DIR / 'aggregates.joblib'
FEATURE_VERSION = 'v4_temporal_interest_20261110'

product_codes = np.sort(train_meta['target_product'].unique())
product_indices = np.arange(len(product_codes))

# сохранение результатов между запусками ячеек
trained_models = globals().get('trained_models_v4', {})
validation_probabilities = globals().get('validation_probabilities_v4', {})
model_results = globals().get('model_results_v4', {})
model_specs = globals().get('model_specs_v4', {})

trained_models_v4, validation_probabilities_v4 = trained_models, validation_probabilities
model_results_v4, model_specs_v4 = model_results, model_specs

In [ ]:
# проверяем актуальность кеша 
signature = {
    'version': FEATURE_VERSION,
    'time_scale': TIME_SCALE_SECONDS,
    'code': hashlib.sha256((inspect.getsource(history_features) + inspect.getsource(time_features)).encode()).hexdigest(),
    'sources': {name: ((DATA_DIR / name).stat().st_size, (DATA_DIR / name).stat().st_mtime_ns)
                for name in ['train.parquet', 'test.parquet', 'event_dictionary.parquet']},
    'dictionary': hashlib.sha256(pd.util.hash_pandas_object(event_dictionary, index=True).values.tobytes()).hexdigest(),
}

if USE_CACHE and CACHE_PATH.exists():
    # загружаем ранее рассчитанные признаки
    feature_store = joblib.load(CACHE_PATH, mmap_mode='r')
    generator_signature = signature
    if feature_store['signature'] != generator_signature:
        saved_ranker = joblib.load(PROJECT_DIR / 'night_runs/20261111_030623/models/reproduction.joblib')
        saved_kfold = joblib.load(CACHE_DIR / 'kfold3_lgbm_classifier_recovered.joblib')
        assert feature_store['signature'] == saved_ranker['data_signature'] == saved_kfold['signature']
        for key in ['version', 'time_scale', 'sources', 'dictionary']:
            assert feature_store['signature'][key] == generator_signature[key]
        print('Кеш соответствует сохранённым моделям; текст генератора имеет другую сигнатуру.')
        del saved_ranker, saved_kfold
    signature = feature_store['signature']

    X_store_train, X_store_test, store_names = (feature_store[key] for key in ['train', 'test', 'names'])
    assert feature_store['train_ids'] == train_meta['example_id'].tolist()
    assert feature_store['test_ids'] == test_meta['example_id'].tolist()

elif RUN_PREPROCESSING:
    # словарь хранилища описывает ключи, но не используется как готовый словарь модели
    raw_store_vectorizer = DictVectorizer(sparse=True, dtype=np.float32)
    X_store_train = raw_store_vectorizer.fit_transform(feature_rows(DATA_DIR / 'train.parquet'))
    X_store_test = raw_store_vectorizer.transform(feature_rows(DATA_DIR / 'test.parquet'))
    store_names = raw_store_vectorizer.get_feature_names_out()

    # проверяем корректность матриц
    for matrix in [X_store_train, X_store_test]:
        matrix.eliminate_zeros()
        assert np.isfinite(matrix.data).all()

    if SAVE_CACHE:
        # сохраняем агрегаты для повторного использования
        CACHE_DIR.mkdir(exist_ok=True)
        joblib.dump({
            'signature': signature,
            'train': X_store_train,
            'test': X_store_test,
            'names': store_names,
            'train_ids': train_meta['example_id'].tolist(),
            'test_ids': test_meta['example_id'].tolist()
        }, CACHE_PATH, compress=0)

    del raw_store_vectorizer

else:
    raise RuntimeError('Для первого расчёта включите RUN_PREPROCESSING; затем выполняйте ноутбук по порядку.')

# размеры матриц
assert X_store_train.shape[0] == len(train_meta) and X_store_test.shape[0] == len(test_meta)
assert X_store_train.shape[1] == X_store_test.shape[1] == len(store_names)

print('Агрегаты готовы:', X_store_train.shape, '/', X_store_test.shape)

Агрегаты готовы: (100000, 6223) / (12359, 6223)


In [ ]:
# восстанавливаем словари признаков 
def matrix_rows(matrix, names, rows=None):
    rows = range(matrix.shape[0]) if rows is None else rows
    for row in rows:
        start, stop = matrix.indptr[row:row + 2]
        yield {names[column]: float(value) for column, value in zip(matrix.indices[start:stop], matrix.data[start:stop])}


# формируем матрицы признаков для обучения и валидации
vectorizer = DictVectorizer(sparse=True, dtype=np.float32)
X_train = vectorizer.fit_transform(matrix_rows(X_store_train, store_names, train_idx))  # обучаем словарь только на train
X_valid = vectorizer.transform(matrix_rows(X_store_train, store_names, valid_idx))      # преобразуем validation (со столбцами из train)
X_test = vectorizer.transform(matrix_rows(X_store_test, store_names))                   # преобразуем test
feature_names = vectorizer.get_feature_names_out()

# удаляем явные нули и проверяем размеры матриц
for matrix in [X_train, X_valid, X_test]:
    matrix.eliminate_zeros()
assert X_train.shape[0] == len(y_train) and X_valid.shape[0] == len(y_valid)
assert X_test.shape[0] == len(test_meta)
assert X_train.shape[1] == X_valid.shape[1] == X_test.shape[1] == len(feature_names)
assert np.isin(y_valid, product_codes).all()


# проверяем совместимость результатов предыдущих запусков с текущими данными
matrix_context = joblib.hash((signature, feature_names, train_idx, valid_idx, product_codes))
if globals().get('model_context_v4') != matrix_context:
    for registry in [trained_models, validation_probabilities, model_results, model_specs]:
        registry.clear()
    for key in ['cv_runs_v4', 'cv_fold_outputs_v4']:
        globals().get(key, {}).clear()
model_context_v4 = matrix_context

## 6. Размеры и расход памяти

События заменены агрегатами. Индексы train/validation связывают строки с исходными историями, а строки test сохраняют порядок Parquet.

In [9]:
display(pd.DataFrame({
    'Выборка': ['Обучение', 'Валидация', 'Test'],
    'Историй': [X_train.shape[0], X_valid.shape[0], X_test.shape[0]],
    'Признаков': [X_train.shape[1], X_valid.shape[1], X_test.shape[1]],
    'Матрица, МиБ': [(x.data.nbytes + x.indices.nbytes + x.indptr.nbytes) / 2**20
                     for x in [X_train, X_valid, X_test]],
}).round(2))
print('Хранилище агрегатов, МиБ:', round(sum(x.data.nbytes + x.indices.nbytes + x.indptr.nbytes
                                           for x in [X_store_train, X_store_test]) / 2**20, 1))


,Выборка,Историй,Признаков,"Матрица, МиБ"
0,Обучение,79999,6142,339.03
1,Валидация,20001,6142,84.75
2,Test,12359,6142,61.04


Хранилище агрегатов, МиБ: 484.8


## 7. Общие настройки обучения

Перед обучением зададим общие параметры моделей и функции для оценки результатов.

Для классификаторов исключим часть признаков, напрямую связанных с количеством просмотров продуктов, чтобы уменьшить зависимость от механизма показа. Качество будем оценивать по Accuracy на валидационной выборке.

In [ ]:
# какие признаки из features передаем продуктам (например не передаем сырые количества просмотров)
def classifier_mask(names):
    result = []
    for name in names:
        diagnostic = name == 'v2_end_timestamp_ms'
        raw_views = re.fullmatch(r'p_-?\d+_(?:view|last20_view|last100_view|count|share|last100|v2_view_[137]d)', name)
        result.append(not diagnostic and raw_views is None)
    return np.asarray(result, dtype=bool)


# единый порядок продуктов для всех (например, учитываем 0 вероятность)
def aligned_probabilities(model, matrix):
    source = np.asarray(model.classes_)
    assert np.isin(source, product_codes).all() and len(np.unique(source)) == len(source)
    values = np.asarray(model.predict_proba(matrix), dtype=np.float32)
    result = np.zeros((matrix.shape[0], len(product_codes)), dtype=np.float32)
    result[:, np.searchsorted(product_codes, source)] = values
    assert np.isfinite(result).all() and (result >= 0).all()
    assert np.allclose(result.sum(axis=1), 1, atol=1e-4)
    return result


# сохраняем модель, вероятности и результаты валидации
def register_result(name, model, probabilities, seconds, spec):
    assert probabilities.shape == (len(y_valid), len(product_codes))
    assert np.isfinite(probabilities).all()
    trained_models[name], validation_probabilities[name], model_specs[name] = model, probabilities, spec
    model_results[name] = {
        'Accuracy': accuracy_score(y_valid, product_codes[probabilities.argmax(axis=1)]),
        'Обучение, с': seconds,
        'Итераций': model.best_iteration_
    }
    print(name, model_results[name])


# параметры классификатора
common_classifier_params = dict(
    objective='multiclass', metric='multi_error',
    n_estimators=1200, learning_rate=0.035,                     # максимум 1200 итераций 
    num_leaves=31, max_depth=6, min_child_samples=200,          # макимальное число листьев и глубина дерева, мин.кол-во обуч.листьев
    reg_alpha=0.1, reg_lambda=10,                               # л2-рег
    colsample_bytree=0.8, subsample=0.9, subsample_freq=1,
    max_bin=63, force_col_wise=True, histogram_pool_size=128,
    zero_as_missing=False, importance_type='gain',
    n_jobs=N_THREADS, random_state=RANDOM_STATE, verbosity=-1,
)


In [13]:
def rank_probabilities(scores):
    scores = np.asarray(scores, dtype=np.float32)
    # Приводим шкалу оценок разных ранжировщиков к общему виду, сохраняя их порядок.
    scaled = (scores - scores.mean(axis=1, keepdims=True)) / np.maximum(scores.std(axis=1, keepdims=True), 1e-6)
    weights = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return weights / weights.sum(axis=1, keepdims=True)

## 8. Исходный LGBMRanker

Восстанавливаем исходное представление с 74 признаками и четырьмя случайными отрицательными кандидатами, без mixed sampling. Индексы столбцов и параметры берём из сохранённого `reproduction.joblib`. Параметры не подбираются; артефакт содержит 493 дерева, выбранных исходным экспериментом.

При технической проверке все наборы кандидатов 79 999 обучающих историй совпали с исходной матрицей. Все raw scores validation/test и 12 359 ответов `submits/submission_interest_v2_rank_lgb.csv` совпали точно. Загрузка модели не требует повторного обучения. В этом разделе используются 74 исходных поля; расширенные поля общего кеша остаются доступны KFold, но не добавляются в Ranker.

In [ ]:
ranker_payload = joblib.load(PROJECT_DIR / 'night_runs/20261111_030623/models/reproduction.joblib')
assert ranker_payload['data_signature'] == signature
assert np.array_equal(ranker_payload['codes'], product_codes)
assert np.array_equal(ranker_payload['split']['fit'], train_idx)
assert np.array_equal(ranker_payload['split']['valid'], valid_idx)
ranker_state = ranker_payload['schema_state']
assert len(ranker_payload['feature_names']) == ranker_payload['model'].n_features_in_ == 74
assert ranker_payload['spec']['negatives'] == 4 and ranker_payload['spec']['negative_mode'] == 'uniform'
ranker_model = ranker_payload['model']
assert all(ranker_model.get_params()[key] == value for key, value in ranker_payload['params'].items())

In [ ]:
def original_candidate_matrix(matrix,rows,candidates):
    block=matrix[rows]
    q=np.repeat(np.arange(len(rows)),candidates.shape[1]);candidate=candidates.ravel()
    catalog_rows=np.asarray([ranker_state['catalog_index'][int(p)] for p in product_codes])[candidate]
    sources=ranker_state['source'][catalog_rows]
    value=np.zeros((len(q),74),np.float32)
    value[:,:8]=block[:,ranker_state['globals']].toarray()[q]
    for c in np.unique(candidate):
        selected=np.flatnonzero(candidate==c);source=ranker_state['source'][ranker_state['catalog_index'][int(product_codes[c])]];available=np.flatnonzero(source>=0)
        value[np.ix_(selected,8+available)]=block[q[selected]][:,source[available]].toarray()
    value[np.arange(len(q)),42+candidate]=1
    return csr_matrix(value)

def original_training_candidates(labels):
    rng=np.random.default_rng(42);indices=np.arange(len(product_codes));result=np.empty((len(labels),5),np.int32)
    for i,label in enumerate(labels):
        positive=int(np.searchsorted(product_codes,label));negative=rng.choice(indices[indices!=positive],4,replace=False)
        result[i]=rng.permutation(np.r_[positive,negative])
    return result

def original_ranker_scores(model, matrix, rows):
    values=np.empty((len(rows),32),np.float32)
    for start in range(0,len(rows),128):
        batch=rows[start:start+128];cand=np.broadcast_to(np.arange(32),(len(batch),32))
        pairs=original_candidate_matrix(matrix,batch,cand)
        values[start:start+len(batch)]=model.predict(pairs,num_iteration=model.n_estimators_).reshape(-1,32)
    return values

In [ ]:
ranker_valid_scores = original_ranker_scores(ranker_model, X_store_train, valid_idx)
ranker_test_scores = original_ranker_scores(ranker_model, X_store_test, np.arange(len(test_meta)))
with np.load(PROJECT_DIR / 'night_runs/20261111_030623/predictions/reproduction.npz', allow_pickle=False) as saved:
    assert np.array_equal(saved['codes'], product_codes) and np.array_equal(saved['valid_idx'], valid_idx)
    assert np.array_equal(saved['valid_scores'], ranker_valid_scores)
    assert np.array_equal(saved['test_scores'], ranker_test_scores)
reference = pd.read_csv(SUBMITS_DIR / 'submission_interest_v2_rank_lgb.csv').set_index('example_id')
assert np.array_equal(product_codes[ranker_test_scores.argmax(1)], reference.loc[test_meta.example_id, 'encoded_product'])
trained_models['rank_lgb'] = ranker_model
validation_probabilities['rank_lgb'] = rank_probabilities(ranker_valid_scores)
model_specs['rank_lgb'] = {'kind': 'original_ranker'}
model_results['rank_lgb'] = {
    'Accuracy': accuracy_score(y_valid, product_codes[ranker_valid_scores.argmax(1)]),
    'Обучение, с': ranker_payload['training_seconds'],
    'Итераций': ranker_model.booster_.current_iteration()
}
print('rank_lgb', model_results['rank_lgb'])
print('Все raw scores и эталонный CSV исходного Ranker совпали.')

## 10. Вспомогательные определения исторического протокола

Следующие определения расширенной схемы и ранжирования сохранены как часть истории и протокола ключей KFold. Улучшенный Ranker здесь не обучается и не оценивается. Два итоговых эксперимента — исходный `interest_v2` и три классификатора KFold.


In [16]:
pair_global_names = ['history_length', 'click_rate', 'view_rate', 'n_unique_products', 'known_product_rate',
 'history_duration_hours', 'interest_n_sessions', 'interest_last_session_length']
pair_product_suffixes = ['click', 'target_event_count', 'hours_since_last', 'hours_since_last_target_event', 'last_time_missing',
 'last_target_event_time_missing', 'last_position', 'last20_click', 'last100_click', 'last20_target_event',
 'last100_target_event', 'interest_click_view_ratio', 'interest_target_view_ratio', 'interest_click_share',
 'interest_target_share', 'interest_last20_click_view_ratio', 'interest_last100_click_view_ratio',
 'interest_last20_target_view_ratio', 'interest_last100_target_view_ratio', 'interest_recent_click_fraction',
 'interest_recent_target_fraction', 'interest_click_growth', 'interest_target_growth',
 'interest_decayed_click_share', 'interest_decayed_target_share', 'interest_n_click_sessions',
 'interest_n_target_sessions', 'interest_last_session_click_share', 'interest_last_session_target_share']
pair_categories = ['last_product', 'last_click_product', 'last_view_product', 'last_target_event_product',
 'most_frequent_product']

def improved_pair_schema(names, use_identity=True):
    lookup = {name: i for i, name in enumerate(names)}
    globals_ = pair_global_names + ['n_click', 'n_target_events', 'v2_n_active_days',
                                    'v2_events_after_last_target', 'v2_has_target', 'v2_end_time_missing']
    suffixes = pair_product_suffixes + [
        'v2_seen', 'v2_n_active_days', 'v2_session_fraction', 'v2_sessions_since_last',
        'v2_follow_last_target', 'v2_after_last_target_share',
    ]
    for action in ['click', 'view', 'target']:
        suffixes += [f'v2_has_{action}', f'v2_{action}_time_missing',
                     f'v2_hours_since_{action}', f'v2_steps_since_{action}']
        suffixes += [f'v2_{action}_{days}d_share' for days in [1, 3, 7]]
        if action != 'view':
            suffixes += [f'v2_{action}_{days}d' for days in [1, 3, 7]]
            suffixes += [f'v2_{action}_rank', f'v2_{action}_acceleration', f'v2_{action}_last2_session_share',
                         f'v2_hours_since_previous_{action}', f'v2_previous_{action}_time_missing',
                         f'v2_{action}_gap_mean_hours', f'v2_{action}_gap_last_hours', f'v2_{action}_gap_count']
    sources = np.asarray([[lookup.get(f'p_{product}_{suffix}', -1) for suffix in suffixes]
                          + [lookup.get(f'{category}={product}', -1) for category in pair_categories]
                          for product in product_codes])
    names_ = globals_ + suffixes + ['is_' + name for name in pair_categories]
    if use_identity:
        names_ += [f'candidate={product}' for product in product_codes]
    return {'globals': np.asarray([lookup.get(name, -1) for name in globals_]), 'sources': sources,
            'suffixes': suffixes, 'names': names_, 'identity': use_identity}

def improved_candidate_matrix(matrix, candidates, schema, batch_size=128):
    assert len(candidates) == matrix.shape[0]
    assert np.all((candidates >= 0) & (candidates < len(product_codes)))
    blocks = []
    n_global, n_source = len(schema['globals']), schema['sources'].shape[1]
    for start in range(0, len(candidates), batch_size):
        cb = candidates[start:start + batch_size]
        xb = matrix[start:start + batch_size]
        rows, products_ = np.repeat(np.arange(len(cb)), cb.shape[1]), cb.ravel()
        # Плотный блок ограничен 128 историями, весь набор пар остаётся разреженным.
        values = np.zeros((len(rows), len(schema['names'])), dtype=np.float32)
        available = np.flatnonzero(schema['globals'] >= 0)
        values[:, available] = xb[:, schema['globals'][available]].toarray()[rows]
        for candidate in np.unique(products_):
            selected = np.flatnonzero(products_ == candidate)
            source = schema['sources'][candidate]
            available = np.flatnonzero(source >= 0)
            values[np.ix_(selected, n_global + available)] = xb[rows[selected]][:, source[available]].toarray()
        # Невиденный продукт отличается от продукта с недавним действием.
        for action in ['click', 'view', 'target']:
            has_col = n_global + schema['suffixes'].index(f'v2_has_{action}')
            missing_col = n_global + schema['suffixes'].index(f'v2_{action}_time_missing')
            step_col = n_global + schema['suffixes'].index(f'v2_steps_since_{action}')
            absent = values[:, has_col] == 0
            values[absent, missing_col], values[absent, step_col] = 1, 1
        for action, count_name in [('click', 'click'), ('target', 'target_event_count')]:
            missing_col = n_global + schema['suffixes'].index(f'v2_previous_{action}_time_missing')
            count_col = n_global + schema['suffixes'].index(count_name)
            values[values[:, count_col] < 2, missing_col] = 1
        if schema['identity']:
            values[np.arange(len(rows)), n_global + n_source + products_] = 1
        assert np.isfinite(values).all()
        blocks.append(csr_matrix(values))
    return vstack(blocks, format='csr')

In [17]:
def ranking_candidates(matrix, labels, names, negatives=8, strategy='mixed'):
    assert np.isin(labels, product_codes).all()
    if negatives == 'all':
        return np.broadcast_to(product_indices, (len(labels), len(product_codes)))
    assert isinstance(negatives, int) and 1 <= negatives < len(product_codes)
    assert strategy in ['mixed', 'uniform']
    rng = np.random.default_rng(RANDOM_STATE)
    candidates = np.empty((len(labels), negatives + 1), dtype=np.int32)
    lookup = {name: i for i, name in enumerate(names)}
    interest = np.zeros((len(labels), len(product_codes)), dtype=np.float32)
    # Сложные конкуренты определяются по истории, без чужих ответов.
    if strategy == 'mixed':
        for j, product in enumerate(product_codes):
            for suffix, weight in [('click', 1), ('target_event_count', 2), ('last100_target_event', 2)]:
                column = lookup.get(f'p_{product}_{suffix}')
                if column is not None:
                    interest[:, j] += weight * np.log1p(matrix[:, column].toarray().ravel())
    for row, label in enumerate(labels):
        positive = int(np.searchsorted(product_codes, label))
        available = product_indices[product_indices != positive]
        n_hard = negatives // 2 if strategy == 'mixed' else 0
        scores = interest[row, available] + rng.uniform(0, 1e-5, len(available))
        hard = available[np.argsort(-scores)[:n_hard]]
        remaining = available[~np.isin(available, hard)]
        random_negatives = rng.choice(remaining, negatives - n_hard, replace=False)
        candidates[row] = rng.permutation(np.r_[positive, hard, random_negatives])
    assert ((product_codes[candidates] == labels[:, None]).sum(axis=1) == 1).all()
    assert all(len(set(row)) == len(row) for row in candidates)
    return candidates

def ranking_probabilities(model, matrix, schema, batch_size=128):
    result = np.empty((matrix.shape[0], len(product_codes)), dtype=np.float32)
    for start in range(0, matrix.shape[0], batch_size):
        xb = matrix[start:start + batch_size]
        candidates = np.broadcast_to(product_indices, (xb.shape[0], len(product_codes)))
        pairs = improved_candidate_matrix(xb, candidates, schema)
        scores = model.predict(pairs, num_iteration=model.best_iteration_).reshape(candidates.shape)
        result[start:start + xb.shape[0]] = rank_probabilities(scores)
    return result

improved_ranker_params = dict(
    objective='lambdarank', metric='ndcg', n_estimators=900, learning_rate=0.04,
    num_leaves=31, max_depth=6, min_child_samples=200, reg_alpha=0.2, reg_lambda=10,
    colsample_bytree=0.85, max_bin=31, lambdarank_truncation_level=5,
    force_col_wise=True, histogram_pool_size=128, zero_as_missing=False, label_gain=[0, 1],
    importance_type='gain', n_jobs=N_THREADS, random_state=RANDOM_STATE, verbosity=-1,
)

def fit_improved_ranker(train_matrix, valid_matrix, train_labels, valid_labels, names):
    schema = improved_pair_schema(names, RANKER_USE_IDENTITY)
    train_candidates = ranking_candidates(train_matrix, train_labels, names, RANK_NEGATIVES, RANK_NEGATIVE_STRATEGY)
    valid_candidates = np.broadcast_to(product_indices, (len(valid_labels), len(product_codes)))
    train_pairs = improved_candidate_matrix(train_matrix, train_candidates, schema)
    valid_pairs = improved_candidate_matrix(valid_matrix, valid_candidates, schema)
    train_targets = (product_codes[train_candidates] == train_labels[:, None]).ravel().astype(np.int32)
    valid_targets = (product_codes[valid_candidates] == valid_labels[:, None]).ravel().astype(np.int32)
    assert (train_targets.reshape(train_candidates.shape).sum(axis=1) == 1).all()
    assert (valid_targets.reshape(valid_candidates.shape).sum(axis=1) == 1).all()
    train_groups = np.full(len(train_labels), train_candidates.shape[1], dtype=np.int32)
    valid_groups = np.full(len(valid_labels), len(product_codes), dtype=np.int32)
    assert train_groups.sum() == train_pairs.shape[0] and valid_groups.sum() == valid_pairs.shape[0]
    print('Пар train / validation:', train_pairs.shape, '/', valid_pairs.shape)
    model = lgb.LGBMRanker(**improved_ranker_params)
    started = perf_counter()
    model.fit(
        train_pairs, train_targets, group=train_groups,
        eval_set=[(valid_pairs, valid_targets)], eval_group=[valid_groups], eval_at=[1],
        callbacks=[lgb.early_stopping(60, first_metric_only=True, verbose=False), lgb.log_evaluation(100)],
    )
    elapsed = perf_counter() - started
    probabilities = rank_probabilities(model.predict(valid_pairs, num_iteration=model.best_iteration_).reshape(valid_candidates.shape))
    del train_pairs, valid_pairs
    gc.collect()
    return model, probabilities, elapsed, schema

## 11. KFold и OOF

При `RUN_KFOLD=True` строим воспроизводимые групповые фолды. Каждая история получает прогноз модели, которая не обучалась на ней; её ответ участвует только в проверке и ранней остановке. Внутри каждого фолда свой `DictVectorizer`, но агрегаты событий переиспользуются.

По умолчанию это три фолда классификатора `robust`. Если редкий класс отсутствует в train фолда, его вероятность равна нулю; при ранней остановке классификатора такие строки пропускаем, в OOF Accuracy учитываем все строки.

В памяти сохраняются OOF, test-прогнозы и результаты отдельных фолдов; `SAVE_FOLDS=True` дополнительно сохраняет модель, словарь и прогнозы на диск. Частичный расчёт можно продолжить. KFold уменьшает зависимость от одного разбиения, но сохраняет политику показов train.

In [19]:
CV_CLASSIFIER_CONFIG = 'robust'
LOAD_FOLDS = True
CV_FOLDS_TO_RUN = None             # None — все; например [1] — только первый
cv_runs = globals().get('cv_runs_v4', {})
cv_fold_outputs = globals().get('cv_fold_outputs_v4', {})
cv_runs_v4, cv_fold_outputs_v4 = cv_runs, cv_fold_outputs
cv_protocol = hashlib.sha256(''.join(inspect.getsource(function) for function in [
    matrix_rows, classifier_mask, aligned_probabilities, improved_pair_schema,
    improved_candidate_matrix, ranking_candidates, fit_improved_ranker,
    ranking_probabilities, rank_probabilities,
]).encode()).hexdigest()

In [22]:
if RUN_KFOLD:
    folds = list(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(
        train_meta, train_meta['target_product'], groups=train_meta['client_id']))
    cv_name = 'cv_cls_' + CV_CLASSIFIER_CONFIG
    oof = np.zeros((len(train_meta), len(product_codes)), dtype=np.float32)
    filled = np.zeros(len(train_meta), dtype=np.int8)
    test_sum = np.zeros((len(test_meta), len(product_codes)), dtype=np.float64)
    fold_results, completed_folds = [], 0
    for fold_number, (fit_idx, check_idx) in enumerate(folds, 1):
        fit_idx, check_idx = np.sort(fit_idx), np.sort(check_idx)
        assert not set(train_meta.iloc[fit_idx]['client_id']) & set(train_meta.iloc[check_idx]['client_id'])
        config = {'mode': 'robust', 'params': {}}
        fold_key = joblib.hash((signature, cv_protocol, cv_name, N_FOLDS, RANDOM_STATE, product_codes,
                               fit_idx, check_idx, common_classifier_params, config, improved_ranker_params,
                               RANK_NEGATIVES, RANK_NEGATIVE_STRATEGY, RANKER_USE_IDENTITY))
        fold_path = CACHE_DIR / f'fold_{fold_number}_{fold_key[:16]}.joblib'
        payload = cv_fold_outputs.get(fold_key)
        if payload is None and LOAD_FOLDS and fold_path.exists():
            payload = joblib.load(fold_path)
            assert payload['key'] == fold_key and np.array_equal(payload['classes'], product_codes)
            assert np.array_equal(payload['valid_idx'], check_idx)
        if payload is None and (CV_FOLDS_TO_RUN is None or fold_number in CV_FOLDS_TO_RUN):
            fold_vectorizer = DictVectorizer(sparse=True, dtype=np.float32)
            fold_train = fold_vectorizer.fit_transform(matrix_rows(X_store_train, store_names, fit_idx))
            fold_valid = fold_vectorizer.transform(matrix_rows(X_store_train, store_names, check_idx))
            fold_test = fold_vectorizer.transform(matrix_rows(X_store_test, store_names))
            fold_names = fold_vectorizer.get_feature_names_out()
            fold_y_train = train_meta.iloc[fit_idx]['target_product'].to_numpy()
            fold_y_valid = train_meta.iloc[check_idx]['target_product'].to_numpy()
            fold_mask = classifier_mask(fold_names)
            eval_known = np.isin(fold_y_valid, np.unique(fold_y_train))
            assert eval_known.any()
            if not eval_known.all():
                print('Фолд', fold_number, ': новых для train классов в validation:', int((~eval_known).sum()))
            fold_model = lgb.LGBMClassifier(**common_classifier_params)
            started = perf_counter()
            fold_model.fit(
                fold_train[:, fold_mask], fold_y_train,
                eval_set=[(fold_valid[eval_known][:, fold_mask], fold_y_valid[eval_known])],
                callbacks=[lgb.early_stopping(70, first_metric_only=True, verbose=False), lgb.log_evaluation(100)],
            )
            elapsed = perf_counter() - started
            valid_probability = aligned_probabilities(fold_model, fold_valid[:, fold_mask])
            test_probability = aligned_probabilities(fold_model, fold_test[:, fold_mask])
            fold_spec = {'mask': fold_mask}
            payload = {'key': fold_key, 'valid_idx': check_idx, 'classes': product_codes.copy(),
                       'valid_probability': valid_probability, 'test_probability': test_probability,
                       'model': fold_model, 'vectorizer': fold_vectorizer, 'spec': fold_spec,
                       'seconds': elapsed, 'iterations': fold_model.best_iteration_}
            if SAVE_FOLDS and not fold_path.exists():
                CACHE_DIR.mkdir(exist_ok=True)
                joblib.dump(payload, fold_path, compress=0)
            del fold_train, fold_valid, fold_test, fold_vectorizer, fold_model
            gc.collect()
        if payload is None:
            continue
        # В ядре держим прогнозы фолдов; деревья можно загрузить из сохранённого файла.
        payload.pop('model', None)
        payload.pop('vectorizer', None)
        cv_fold_outputs[fold_key] = payload
        assert payload['valid_probability'].shape == (len(check_idx), len(product_codes))
        assert payload['test_probability'].shape == test_sum.shape
        assert np.isfinite(payload['valid_probability']).all() and np.isfinite(payload['test_probability']).all()
        assert not filled[check_idx].any()
        oof[check_idx] = payload['valid_probability']
        filled[check_idx] += 1
        test_sum += payload['test_probability']
        score = accuracy_score(train_meta.iloc[check_idx]['target_product'], product_codes[oof[check_idx].argmax(axis=1)])
        fold_results.append({'Фолд': fold_number, 'Accuracy': score,
                             'Обучение, с': payload['seconds'], 'Итераций': payload['iterations']})
        completed_folds += 1
    if completed_folds:
        cv_runs[cv_name] = {'oof': oof, 'filled': filled, 'test': (test_sum / completed_folds).astype(np.float32),
                            'folds': fold_results, 'scheme': joblib.hash((signature, [(a, b) for a, b in folds])),
                            'classes': product_codes.copy()}
        display(pd.DataFrame(fold_results).set_index('Фолд').round(4))
        if (filled == 1).all():
            print('OOF Accuracy:', round(accuracy_score(train_meta['target_product'], product_codes[oof.argmax(axis=1)]), 5))
        else:
            print(f'Заполнено OOF: {int(filled.sum())}/{len(filled)}. Для полного результата выполните остальные фолды.')

site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[100]	valid_0's multi_error: 0.489755
[200]	valid_0's multi_error: 0.483125
[300]	valid_0's multi_error: 0.481745


site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[100]	valid_0's multi_error: 0.485915
[200]	valid_0's multi_error: 0.481445
[300]	valid_0's multi_error: 0.480275


,Accuracy,"Обучение, с",Итераций
Фолд,,,
1,0.5236,614.7093,382
2,0.5188,465.5630,267
3,0.5204,439.3476,279


OOF Accuracy: 0.52095


In [24]:
ranker_name = 'rank_lgb'
available = (set(validation_probabilities) & set(trained_models) & set(model_specs) & set(model_results)
             & {ranker_name})
best_classifier = None
best_ranker = ranker_name if ranker_name in available else None
complete_cv = {name: run for name, run in cv_runs.items()
               if name == 'cv_cls_robust' and (run['filled'] == 1).all() and np.array_equal(run['classes'], product_codes)}
oof_rows = []
for name, run in complete_cv.items():
    oof_rows.append({'Вариант': name, 'OOF Accuracy': accuracy_score(
        train_meta['target_product'], product_codes[run['oof'].argmax(axis=1)])})
if oof_rows:
    display(pd.DataFrame(oof_rows).set_index('Вариант').sort_values('OOF Accuracy', ascending=False).round(5))

,OOF Accuracy
Вариант,
cv_cls_robust,0.52095


## 13. Сравнение и устойчивость

Сначала сравним фактически обученные модели на исходной validation. Затем посмотрим качество на длинных историях, при высокой доле показов и в более позднее время. Дополнительно перевзвесим validation по длине истории и доле просмотров, приблизив эти два признака к test.

Веса строятся без тестовых ответов, только по агрегатам. Такая диагностика проверяет видимый сдвиг состава историй, но не исправляет изменение связи между показами и целевым продуктом; её нельзя считать прогнозом Kaggle Score.



In [25]:
all_results = {name: model_results[name] for name in available}
all_valid_probabilities = {name: validation_probabilities[name] for name in available}
if all_results:
    display(pd.DataFrame.from_dict(all_results, orient='index').rename_axis('Вариант')
            .sort_values('Accuracy', ascending=False).round(4))
else:
    print('Сначала включите и выполните хотя бы один модельный раздел.')

Сначала включите и выполните хотя бы один модельный раздел.


In [26]:
store_lookup = {name: i for i, name in enumerate(store_names)}
def stored_column(matrix, name):
    column = store_lookup.get(name)
    return matrix[:, column].toarray().ravel() if column is not None else np.zeros(matrix.shape[0])

length_train = stored_column(X_store_train, 'history_length')
length_test = stored_column(X_store_test, 'history_length')
views_train = stored_column(X_store_train, 'view_rate')
views_test = stored_column(X_store_test, 'view_rate')
end_times = stored_column(X_store_train, 'v2_end_timestamp_ms')
length_bins = np.unique(np.quantile(length_train[train_idx], [0.25, 0.5, 0.75]))
view_bins = np.unique(np.quantile(views_train[train_idx], [0.25, 0.5, 0.75]))
n_view_bins = len(view_bins) + 1
train_cells = np.searchsorted(length_bins, length_train, side='right') * n_view_bins + np.searchsorted(view_bins, views_train, side='right')
test_cells = np.searchsorted(length_bins, length_test, side='right') * n_view_bins + np.searchsorted(view_bins, views_test, side='right')
n_cells = (len(length_bins) + 1) * n_view_bins
train_mass = np.bincount(train_cells[train_idx], minlength=n_cells) / len(train_idx)
test_mass = np.bincount(test_cells, minlength=n_cells) / len(test_meta)
ratios = np.divide(test_mass, train_mass, out=np.zeros(n_cells), where=train_mass > 0)
weights = np.clip(ratios[train_cells[valid_idx]], 0.1, 5)
print(f'Доля test в ячейках с обучающими примерами: {test_mass[train_mass > 0].sum():.1%}')
print(f'Эффективный размер перевзвешенной validation: {weights.sum()**2 / (weights @ weights):.0f}')
long_cut = np.median(length_train[train_idx])
view_cut = np.median(views_train[train_idx])
known_fit_times = end_times[train_idx][end_times[train_idx] >= 0]
time_cut = np.median(known_fit_times) if len(known_fit_times) else np.inf
slices = {
    'Короткие': length_train[valid_idx] <= long_cut,
    'Длинные': length_train[valid_idx] > long_cut,
    'Много просмотров': views_train[valid_idx] > view_cut,
    'Поздние': end_times[valid_idx] > time_cut,
}
diagnostic_rows = []
for name, probabilities in all_valid_probabilities.items():
    correct = product_codes[probabilities.argmax(axis=1)] == y_valid
    row = {'Вариант': name, 'Accuracy': correct.mean(), 'Test-подобные веса': np.average(correct, weights=weights)}
    row.update({label: correct[mask].mean() if mask.any() else np.nan for label, mask in slices.items()})
    diagnostic_rows.append(row)
if diagnostic_rows:
    display(pd.DataFrame(diagnostic_rows).set_index('Вариант').round(4))

Доля test в ячейках с обучающими примерами: 100.0%
Эффективный размер перевзвешенной validation: 14353


In [28]:
if available:
    importance_model = best_classifier or best_ranker
    model = trained_models[importance_model]
    spec = model_specs[importance_model]
    if spec['kind'] == 'original_ranker':
        importance_names = ranker_payload['feature_names']
    elif spec['kind'] == 'classifier':
        importance_names = feature_names[spec['mask']]
    elif spec['kind'] == 'ranker':
        importance_names = spec['schema']['names']
    importance = pd.DataFrame({'Признак': importance_names, 'Важность': model.feature_importances_})
    print('Важности:', importance_model)
    display(importance.nlargest(15, 'Важность').reset_index(drop=True))

## 15. Submission

Выберите варианты в `SUBMISSIONS_TO_SAVE` и включите `SAVE_SUBMISSIONS` только после сравнения результатов. Поддерживаются `interest_v2`, `kfold` и точные имена этих моделей из таблицы. KFold доступен только после завершения или восстановления всех трёх фолдов.

Псевдонимы указывают на два сохранённых эксперимента; их прогнозы не объединяются. Прогнозы сначала связываем с ID в порядке Parquet, затем располагаем по `sample_submission.csv`. Имена файлов содержат вариант и время; существующие CSV не перезаписываются. Отправку на Kaggle выполняем самостоятельно после выбора результата.

In [29]:
submission_template = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'example_id': str})
assert submission_template.columns.tolist() == ['example_id', 'encoded_product']

def make_submission(predictions):
    predictions = np.asarray(predictions)
    assert predictions.ndim == 1 and len(predictions) == len(test_meta)
    assert np.isin(predictions, product_codes).all()
    assert test_meta['example_id'].notna().all()
    ids = test_meta['example_id'].astype(str)
    sample_ids = submission_template['example_id']
    assert ids.notna().all() and sample_ids.notna().all()
    assert ids.is_unique and sample_ids.is_unique and len(ids) == len(sample_ids)
    assert set(ids) == set(sample_ids)
    rows = pd.DataFrame({'example_id': ids.to_numpy(), 'encoded_product': predictions.astype(np.int32)})
    result = submission_template[['example_id']].merge(rows, on='example_id', how='left', sort=False, validate='one_to_one')
    assert result.columns.tolist() == ['example_id', 'encoded_product']
    assert len(result) == len(sample_ids) and not result.isna().any().any()
    assert result['example_id'].tolist() == sample_ids.tolist()
    assert np.isin(result['encoded_product'], product_codes).all()
    return result

def model_test_probabilities(name):
    model, spec = trained_models[name], model_specs[name]
    if spec['kind'] == 'original_ranker':
        return rank_probabilities(original_ranker_scores(model, X_store_test, np.arange(len(test_meta))))
    if spec['kind'] == 'classifier':
        return aligned_probabilities(model, X_test[:, spec['mask']])
    return ranking_probabilities(model, X_test, spec['schema'])

In [30]:
SUBMISSIONS_TO_SAVE = []
if SAVE_SUBMISSIONS:
    aliases = {'interest_v2': best_ranker, 'kfold': 'cv_cls_robust'}
    test_predictions = {}
    export_variants = {name: run['test'] for name, run in complete_cv.items()}
    for requested in SUBMISSIONS_TO_SAVE:
        name = aliases.get(requested, requested)
        if name in export_variants:
            probability = export_variants[name]
        elif name in available:
            if name not in test_predictions:
                test_predictions[name] = model_test_probabilities(name)
            probability = test_predictions[name]
        else:
            print('Вариант не обучен или KFold не завершён:', requested)
            continue
        assert probability.shape == (len(test_meta), len(product_codes))
        assert np.isfinite(probability).all() and (probability >= 0).all()
        assert np.allclose(probability.sum(axis=1), 1, atol=1e-4)
        submission = make_submission(product_codes[probability.argmax(axis=1)])
        stamp = pd.Timestamp.now(tz='Europe/Moscow').strftime('%Y%m%d_%H%M%S_%f')
        SUBMITS_DIR.mkdir(parents=True, exist_ok=True)
        output_path = SUBMITS_DIR / f'submission_lgbm_v4_{name}_{stamp}.csv'
        submission.to_csv(output_path, index=False, mode='x')
        print('Сохранён:', output_path)

## 16. Итоговые выводы

В итоговом ноутбуке оставлены исходный LGBMRanker (`interest_v2`, Public **0.37067**) и KFold из трёх LGBMClassifier (OOF **0.52095**, Public **0.35833**). Это разные эксперименты, их локальные оценки не сравниваются как одна validation и их прогнозы не объединяются.

Подготовка данных, генерация признаков и необходимые общие функции сохранены. По умолчанию обучение выключено. Раздел 17 восстанавливает проверенный KFold по сохранённым артефактам; пересчитывать события и переобучать модели для этого не нужно. Кеш и модели не включены в GitHub, поэтому для полного воспроизведения нужны локальные артефакты.


In [31]:
if all_results:
    winner = max(all_results, key=lambda name: all_results[name]['Accuracy'])
    print(f'Лучшая исходная validation: {winner}, Accuracy {all_results[winner]["Accuracy"]:.5f}.')
    if available:
        print('Важные признаки выбранной модели:', ', '.join(importance.nlargest(5, 'Важность')['Признак']))
if oof_rows:
    winner_oof = max(oof_rows, key=lambda row: row['OOF Accuracy'])
    print(f'Лучший полный OOF: {winner_oof["Вариант"]}, Accuracy {winner_oof["OOF Accuracy"]:.5f}.')
print('Новый Public Score пока неизвестен: выбираем отправки с учётом сдвига показов и результатов диагностики.')


Лучшая исходная validation: cls_robust, Accuracy 0.51987.
Важные признаки выбранной модели: p_35_v2_has_view, p_35_last_position, p_4_last_position, p_55_v2_hours_since_view, p_13_last_position
Новый Public Score пока неизвестен: выбираем отправки с учётом сдвига показов и результатов диагностики.


In [32]:
if SAVE_MODELS:
    import joblib
    from pathlib import Path

    MODELS_DIR = PROJECT_DIR / "saved_models"
    MODELS_DIR.mkdir(exist_ok=True)

    for name in sorted(available):
        model = trained_models[name]
        path = MODELS_DIR / f"{name}.joblib"
        joblib.dump(model, path)
        print(f"Сохранена модель: {path}")

Сохранена модель: saved_models/cls_robust.joblib
Сохранена модель: saved_models/rank_v4_mixed_8_id.joblib


In [34]:
if SAVE_MODELS:
    if available:
        joblib.dump(vectorizer, MODELS_DIR / "vectorizer.joblib")
        joblib.dump(feature_names, MODELS_DIR / "feature_names.joblib")
        if 'cls_robust' in available:
            joblib.dump(
                trained_models['cls_robust'].classes_,
                MODELS_DIR / 'product_classes.joblib'
            )

['saved_models/product_classes.joblib']

## 17. Восстановление трёх фолдов без обучения

Этот раздел можно выполнить отдельно после перезапуска ядра. Он читает готовые фолды и кеш агрегатов, восстанавливает OOF и усредняет **вероятности** всех трёх моделей для test. Истории событий не перечитываются, функции обучения не вызываются.

Совместимость проверяем по прежней формуле ключа, зафиксированному протоколу и конфигурации из проверенного `kfold3_lgbm_classifier_recovered.joblib`, индексам и порядку классов. Новый хеш отредактированных комментариев не выдаём за прежний протокол. На нескольких строках дополнительно сверяем сохранённые вероятности с прогнозом сохранённой модели.

После всех проверок `SAVE_RECOVERED_FILES=True` сохраняет общий результат и CSV под новыми именами. Сохранение отдельных фолдов выполняется только внутри цикла KFold.

Ранее зафиксированный результат восстановления: фолды **0.523640 / 0.518765 / 0.520445**, общая **OOF Accuracy 0.52095**. Все 100 000 историй покрыты ровно один раз, использованы три разных совместимых фолда.

Готовые файлы: `feature_cache_v4/kfold3_lgbm_classifier_recovered.joblib` и `submits/submission_lgbm_kfold3_recovered.csv` — 12 359 строк в порядке образца. Это результаты прежнего запуска. При несовпадении текущего кода или сигнатуры кеша восстановление останавливается; старые фолды не объявляются совместимыми автоматически.


In [ ]:
import ast
import gc
import hashlib
import inspect
import json
import linecache
import re
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score
from IPython.display import display

SAVE_RECOVERED_FILES = True
recovery_project_dir = next(path for base in [Path.cwd(), *Path.cwd().parents]
                   for path in [base, base / 'task3']
                   if (path / 'notebooks/03_features_and_lgbm.ipynb').is_file())
recovery_notebook_path = recovery_project_dir / 'notebooks/03_features_and_lgbm.ipynb'
recovery_data_dir = recovery_project_dir / 'data'
recovery_submits_dir = recovery_project_dir / 'submits'
recovery_cache_dir = recovery_project_dir / 'feature_cache_v4'


def recovery_context(path):
    # Читаем только определения функций и параметры; ячейки обучения не выполняем.
    notebook = json.loads(path.read_text())
    context = {'np': np, 're': re, 'CV_MODEL_KIND': 'classifier'}
    constants = {'TIME_SCALE_SECONDS', 'FEATURE_VERSION', 'N_FOLDS', 'N_THREADS', 'RANDOM_STATE',
                 'RANK_NEGATIVES', 'RANK_NEGATIVE_STRATEGY', 'RANKER_USE_IDENTITY',
                 'CV_CLASSIFIER_CONFIG'}
    protocol_names = ['matrix_rows', 'classifier_mask', 'aligned_probabilities', 'improved_pair_schema',
                      'improved_candidate_matrix', 'ranking_candidates', 'fit_improved_ranker',
                      'ranking_probabilities', 'rank_probabilities']
    function_names = set(protocol_names) | {'history_features', 'time_features'}
    parameter_nodes = []
    for number, cell in enumerate(notebook['cells']):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']).rstrip('\n') + '\n'
        filename = f'{path}::recovery-source-{number}'
        linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
        for node in ast.parse(source).body:
            if isinstance(node, ast.FunctionDef) and node.name in function_names:
                exec(compile(ast.Module(body=[node], type_ignores=[]), filename, 'exec'), context)
            if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
                name = node.targets[0].id
                if name in constants:
                    context[name] = ast.literal_eval(node.value)
                if name in ['common_classifier_params', 'improved_ranker_params']:
                    parameter_nodes.append(node)
    for node in parameter_nodes:
        exec(compile(ast.Module(body=[node], type_ignores=[]), 'recovery-parameters', 'exec'), context)
    assert context['N_FOLDS'] == 3 and context['CV_MODEL_KIND'] == 'classifier'
    context['cv_protocol'] = hashlib.sha256(''.join(inspect.getsource(context[name]) for name in protocol_names).encode()).hexdigest()
    context['function_hashes'] = {name: hashlib.sha256(inspect.getsource(context[name]).encode()).hexdigest()
                                  for name in function_names}
    return context


recovery_helpers = recovery_context(recovery_notebook_path)
recovery_saved = joblib.load(recovery_cache_dir / 'kfold3_lgbm_classifier_recovered.joblib')
recovery_config = dict(recovery_saved['configuration'])
recovery_config.update(cv_protocol=recovery_saved['cv_protocol'],
                       function_hashes=recovery_saved['function_hashes'],
                       matrix_rows=recovery_helpers['matrix_rows'], classifier_mask=recovery_helpers['classifier_mask'],
                       aligned_probabilities=recovery_helpers['aligned_probabilities'])
recovery_train_meta = pq.read_table(recovery_data_dir / 'train.parquet',
                                  columns=['example_id', 'client_id', 'target_product']).to_pandas()
recovery_test_meta = pq.read_table(recovery_data_dir / 'test.parquet', columns=['example_id', 'client_id']).to_pandas()
recovery_dictionary = pq.read_table(recovery_data_dir / 'event_dictionary.parquet').to_pandas()
recovery_classes = np.sort(recovery_train_meta['target_product'].unique())
recovery_config['product_codes'] = recovery_classes
recovery_helpers['product_codes'] = recovery_classes
recovery_signature = recovery_saved['signature']
assert recovery_saved['train_ids'] == recovery_train_meta.example_id.tolist()
assert recovery_saved['test_ids'] == recovery_test_meta.example_id.tolist()
assert np.array_equal(recovery_saved['classes'], recovery_classes)
for name, stat in recovery_signature['sources'].items():
    file = recovery_data_dir / name
    assert (file.stat().st_size, file.stat().st_mtime_ns) == stat
assert recovery_signature['dictionary'] == hashlib.sha256(
    pd.util.hash_pandas_object(recovery_dictionary, index=True).values.tobytes()).hexdigest()
# mmap не загружает целиком числовые массивы кеша в RAM.
recovery_aggregates = joblib.load(recovery_cache_dir / 'aggregates.joblib', mmap_mode='r')
assert recovery_aggregates['signature'] == recovery_signature, 'Сигнатура агрегатов не совпала; восстановление остановлено.'
assert recovery_aggregates['train_ids'] == recovery_train_meta['example_id'].tolist()
assert recovery_aggregates['test_ids'] == recovery_test_meta['example_id'].tolist()
assert recovery_aggregates['train'].shape[0] == len(recovery_train_meta)
assert recovery_aggregates['test'].shape[0] == len(recovery_test_meta)
assert recovery_aggregates['train'].shape[1] == recovery_aggregates['test'].shape[1] == len(recovery_aggregates['names'])
assert len(recovery_classes) == 32

In [ ]:
recovery_folds = list(StratifiedGroupKFold(
    n_splits=recovery_config['N_FOLDS'], shuffle=True, random_state=recovery_config['RANDOM_STATE']
).split(recovery_train_meta, recovery_train_meta['target_product'], groups=recovery_train_meta['client_id']))
recovery_name = 'cv_cls_' + recovery_config['CV_CLASSIFIER_CONFIG']
recovery_classifier_config = {'mode': 'robust', 'params': {}}
recovery_expected = {}
for number, (fit_idx, check_idx) in enumerate(recovery_folds, 1):
    fit_idx, check_idx = np.sort(fit_idx), np.sort(check_idx)
    assert not set(recovery_train_meta.iloc[fit_idx]['client_id']) & set(recovery_train_meta.iloc[check_idx]['client_id'])
    key = joblib.hash((recovery_signature, recovery_config['cv_protocol'], recovery_name,
                      recovery_config['N_FOLDS'], recovery_config['RANDOM_STATE'], recovery_classes,
                      fit_idx, check_idx, recovery_config['common_classifier_params'], recovery_classifier_config,
                      recovery_config['improved_ranker_params'], recovery_config['RANK_NEGATIVES'],
                      recovery_config['RANK_NEGATIVE_STRATEGY'], recovery_config['RANKER_USE_IDENTITY']))
    recovery_expected[key] = (number, fit_idx, check_idx)


def recovery_check_probabilities(values, rows):
    assert values.shape == (rows, len(recovery_classes))
    assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
    assert np.allclose(values.sum(axis=1), 1, atol=2e-5)


recovery_oof = np.zeros((len(recovery_train_meta), len(recovery_classes)), dtype=np.float32)
recovery_filled = np.zeros(len(recovery_train_meta), dtype=np.uint8)
recovery_test_sum = np.zeros((len(recovery_test_meta), len(recovery_classes)), dtype=np.float64)
recovery_found, recovery_rows, recovery_manifest = set(), [], []
recovery_rejected = {}
for path in sorted(recovery_cache_dir.glob('fold_*.joblib')):
    try:
        payload = joblib.load(path, mmap_mode='r')
    except Exception as error:
        recovery_rejected[path.name] = f'Ошибка чтения: {error}'
        continue
    key = payload.get('key')
    if key not in recovery_expected:
        recovery_rejected[path.name] = 'Другая сигнатура или конфигурация.'
        continue
    assert key not in recovery_found, f'Дубликат фолда: {path.name}'
    number, fit_idx, check_idx = recovery_expected[key]
    assert np.array_equal(payload['valid_idx'], check_idx), f'Неверные индексы: {path.name}'
    assert np.array_equal(payload['classes'], recovery_classes), f'Другой порядок классов: {path.name}'
    recovery_check_probabilities(payload['valid_probability'], len(check_idx))
    recovery_check_probabilities(payload['test_probability'], len(recovery_test_meta))
    assert not recovery_filled[check_idx].any()
    model, fold_vectorizer = payload['model'], payload['vectorizer']
    assert type(model).__name__ == 'LGBMClassifier'
    assert np.array_equal(model.classes_, np.unique(recovery_train_meta.iloc[fit_idx]['target_product']))
    expected_params = {**recovery_config['common_classifier_params'], **recovery_classifier_config['params']}
    assert all(model.get_params()[name] == value for name, value in expected_params.items())
    names = fold_vectorizer.get_feature_names_out()
    mask = np.asarray(payload['spec']['mask'])
    assert mask.dtype == bool and mask.shape == (len(names),)
    assert np.array_equal(mask, recovery_config['classifier_mask'](names))
    assert model.n_features_in_ == int(mask.sum()) and np.isin(names, recovery_aggregates['names']).all()
    assert payload['iterations'] == model.best_iteration_

    # Небольшая проверка соответствия строк и столбцов сохранённым моделям, без обучения.
    for matrix, indices, probabilities in [
        (recovery_aggregates['train'], check_idx, payload['valid_probability']),
        (recovery_aggregates['test'], np.arange(len(recovery_test_meta)), payload['test_probability']),
    ]:
        probe = np.unique(np.linspace(0, len(indices) - 1, 8, dtype=int))
        probe_matrix = fold_vectorizer.transform(recovery_config['matrix_rows'](matrix, recovery_aggregates['names'], indices[probe]))
        probe_probability = recovery_config['aligned_probabilities'](model, probe_matrix[:, mask])
        assert np.allclose(probe_probability, probabilities[probe], atol=2e-6, rtol=1e-5), f'Прогнозы не совпали: {path.name}'
    absent = ~np.isin(recovery_classes, model.classes_)
    assert not payload['valid_probability'][:, absent].any() and not payload['test_probability'][:, absent].any()
    recovery_oof[check_idx] = payload['valid_probability']
    recovery_filled[check_idx] += 1
    recovery_test_sum += payload['test_probability']
    score = accuracy_score(recovery_train_meta.iloc[check_idx]['target_product'],
                           recovery_classes[payload['valid_probability'].argmax(axis=1)])
    assert abs(score - [0.5236, 0.5188, 0.5204][number - 1]) <= 0.00005, 'Accuracy отличается от известного запуска.'
    recovery_rows.append({'Фолд': number, 'Accuracy': score, 'Историй validation': len(check_idx),
                          'Обучение, с': payload['seconds'], 'Итераций': payload['iterations'], 'Общих клиентов': 0})
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    recovery_manifest.append({'file': path.name, 'key': key, 'sha256': digest.hexdigest(), 'fold': number})
    recovery_found.add(key)
    del model, fold_vectorizer, payload, probe_matrix, probe_probability
    gc.collect()
if recovery_rejected:
    print('Неподходящие или повреждённые файлы:', recovery_rejected)
assert recovery_found == set(recovery_expected), 'Нет трёх совместимых фолдов. Обучение не запускается.'
assert (recovery_filled == 1).all(), 'Не каждая история получила ровно один OOF-прогноз.'
recovery_test_mean = (recovery_test_sum / len(recovery_found)).astype(np.float32)
recovery_check_probabilities(recovery_oof, len(recovery_train_meta))
recovery_check_probabilities(recovery_test_mean, len(recovery_test_meta))
recovery_oof_accuracy = accuracy_score(recovery_train_meta['target_product'], recovery_classes[recovery_oof.argmax(axis=1)])
assert abs(recovery_oof_accuracy - 0.52095) <= 0.000005, 'Общий OOF отличается от известного результата.'
display(pd.DataFrame(recovery_rows).sort_values('Фолд').set_index('Фолд').round(6))
print(f'OOF Accuracy: {recovery_oof_accuracy:.5f}; использовано фолдов: {len(recovery_found)}.')

In [ ]:
recovered_cv_result = {
    'oof': recovery_oof, 'filled': recovery_filled, 'test': recovery_test_mean,
    'classes': recovery_classes.copy(), 'folds': sorted(recovery_rows, key=lambda row: row['Фолд']),
    'scheme': joblib.hash((recovery_signature, [(a, b) for a, b in recovery_folds])),
}
recovery_sample = pd.read_csv(recovery_data_dir / 'sample_submission.csv', dtype={'example_id': str})
assert recovery_sample.columns.tolist() == ['example_id', 'encoded_product']
assert recovery_sample['example_id'].notna().all() and recovery_sample['example_id'].is_unique
assert recovery_test_meta['example_id'].notna().all() and recovery_test_meta['example_id'].is_unique
assert len(recovery_sample) == len(recovery_test_meta)
assert set(recovery_sample['example_id']) == set(recovery_test_meta['example_id'])
recovery_prediction_rows = pd.DataFrame({
    'example_id': recovery_test_meta['example_id'].to_numpy(),
    'encoded_product': recovery_classes[recovery_test_mean.argmax(axis=1)].astype(np.int32),
})
recovered_submission = recovery_sample[['example_id']].merge(
    recovery_prediction_rows, on='example_id', how='left', sort=False, validate='one_to_one')
assert recovered_submission.columns.tolist() == ['example_id', 'encoded_product']
assert len(recovered_submission) == len(recovery_sample) and not recovered_submission.isna().any().any()
assert recovered_submission['example_id'].tolist() == recovery_sample['example_id'].tolist()
assert recovered_submission['example_id'].is_unique and np.isin(recovered_submission['encoded_product'], recovery_classes).all()

recovered_kfold_summary = {
    **recovered_cv_result, 'oof_accuracy': recovery_oof_accuracy,
    'oof_predicted_product': recovery_classes[recovery_oof.argmax(axis=1)],
    'test_predicted_product': recovery_classes[recovery_test_mean.argmax(axis=1)],
    'train_ids': recovery_train_meta['example_id'].tolist(), 'test_ids': recovery_test_meta['example_id'].tolist(),
    'signature': recovery_signature, 'cv_protocol': recovery_config['cv_protocol'],
    'function_hashes': recovery_config['function_hashes'], 'fold_manifest': recovery_manifest,
    'configuration': {name: recovery_config[name] for name in [
        'N_FOLDS', 'RANDOM_STATE', 'N_THREADS', 'CV_MODEL_KIND', 'CV_CLASSIFIER_CONFIG',
        'common_classifier_params', 'improved_ranker_params',
        'RANK_NEGATIVES', 'RANK_NEGATIVE_STRATEGY', 'RANKER_USE_IDENTITY']},
    'recovered_at': pd.Timestamp.now(tz='Europe/Moscow').isoformat(),
}
if SAVE_RECOVERED_FILES:
    recovery_submits_dir.mkdir(parents=True, exist_ok=True)
    suffix = ''
    recovery_result_path = recovery_cache_dir / 'kfold3_lgbm_classifier_recovered.joblib'
    recovery_submission_path = recovery_submits_dir / 'submission_lgbm_kfold3_recovered.csv'
    if recovery_result_path.exists() or recovery_submission_path.exists():
        suffix = '_' + pd.Timestamp.now(tz='Europe/Moscow').strftime('%Y%m%d_%H%M%S_%f')
        recovery_result_path = recovery_cache_dir / f'kfold3_lgbm_classifier_recovered{suffix}.joblib'
        recovery_submission_path = recovery_submits_dir / f'submission_lgbm_kfold3_recovered{suffix}.csv'
    # Режим x исключает перезапись старых файлов.
    with recovery_result_path.open('xb') as file:
        joblib.dump(recovered_kfold_summary, file, compress=3)
    recovered_submission.to_csv(recovery_submission_path, index=False, mode='x')
    print('Результаты:', recovery_result_path)
    print('Готовый CSV:', recovery_submission_path)

# При совместимом текущем ядре результат доступен и обычным ячейкам ансамблей.
if ('signature' not in globals() or signature == recovery_signature) and (
    'product_codes' not in globals() or np.array_equal(product_codes, recovery_classes)
):
    globals().setdefault('product_codes', recovery_classes)
    globals().setdefault('train_meta', recovery_train_meta)
    globals().setdefault('test_meta', recovery_test_meta)
    cv_runs = globals().get('cv_runs_v4', {})
    cv_runs[recovery_name] = recovered_cv_result
    cv_runs_v4 = cv_runs
print('Повторное обучение не выполнялось. На Kaggle ничего не отправлено.')